## Your objective
engineer linguistic/statistical features that distinguish human from AI text, analyze which patterns differ between the two classes, and build a reusable function that takes any text and returns its feature values
1. Extract features per text row: word count, sentence count, average sentence length, sentence-length variance (burstiness), average word length, type-token ratio (vocabulary diversity), punctuation frequency, stopword ratio
2. Compare feature distributions between author=human and author=ai (histograms/boxplots) to show which features actually separate the classes
3. Wrap this into a function extract_features(text) that returns a dict of these values for any single input text, so  others can reuse it
4. Save the featured dataset as data/processed/featured_dataset.csv for the model training step

**This feeds directly into the shared baseline model (already trained on TF-IDF alone, but your feature analysis becomes report evidence — e.g., "AI text shows X% lower burstiness"). Even if the model doesn't use every feature, the comparison charts are valuable for the Results/Discussion section**

In [ ]:
import os, re, math
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

INPUT = "data/processed/final_training_dataset.csv"
OUTPUT = "data/processed/featured_dataset.csv"
PLOT_DIR = "data/processed/feature_plots"

os.makedirs(os.path.dirname(OUTPUT), exist_ok=True)
os.makedirs(PLOT_DIR, exist_ok=True)

# Lightweight English stopword list: avoids NLTK downloads and keeps this CPU-only.
STOPWORDS = {
    "a","an","and","are","as","at","be","been","but","by","for","from","had","has",
    "have","he","her","hers","him","his","i","if","in","into","is","it","its","me",
    "my","of","on","or","our","ours","she","so","that","the","their","theirs","them",
    "then","there","these","they","this","those","to","too","was","we","were","what",
    "when","where","which","who","why","will","with","you","your","yours","do","does",
    "did","not","no","nor","can","could","should","would","may","might","must"
}

WORD_RE = re.compile(r"\b[\w]+(?:['’\-][\w]+)*\b", re.UNICODE)
SENTENCE_RE = re.compile(r"(?<=[.!?])(?:[\"'”’)\]]+)?\s+|(?<=[.!?])$")

def extract_features(text):
    """Return linguistic/statistical features for one text string."""
    if pd.isna(text):
        text = ""
    text = str(text).strip()

    words = WORD_RE.findall(text.lower())
    word_count = len(words)

    # Sentence segmentation without external NLP libraries.
    raw_sentences = [s.strip() for s in SENTENCE_RE.split(text) if s.strip()]
    if not raw_sentences and text:
        raw_sentences = [text]

    sentence_count = len(raw_sentences)
    sentence_lengths = [len(WORD_RE.findall(s)) for s in raw_sentences]
    avg_sentence_length = (
        float(np.mean(sentence_lengths)) if sentence_lengths else 0.0
    )
    sentence_length_variance = (
        float(np.var(sentence_lengths)) if sentence_lengths else 0.0
    )

    avg_word_length = (
        float(np.mean([len(w.replace("'", "").replace("’", "").replace("-", ""))
                       for w in words]))
        if words else 0.0
    )

    type_token_ratio = (
        len(set(words)) / word_count if word_count else 0.0
    )

    # Punctuation frequency = punctuation characters per word.
    punctuation_count = sum(1 for ch in text if ch in r"""!"#$%&'()*+,-./:;<=>?@[\]^_`{|}~""")
    punctuation_frequency = punctuation_count / word_count if word_count else 0.0

    stopword_count = sum(1 for w in words if w in STOPWORDS)
    stopword_ratio = stopword_count / word_count if word_count else 0.0

    return {
        "word_count": word_count,
        "sentence_count": sentence_count,
        "average_sentence_length": avg_sentence_length,
        "sentence_length_variance": sentence_length_variance,
        "average_word_length": avg_word_length,
        "type_token_ratio": type_token_ratio,
        "punctuation_frequency": punctuation_frequency,
        "stopword_ratio": stopword_ratio,
    }

# Load the teammate's processed dataset.
df = pd.read_csv(INPUT)

required_columns = {"text", "author"}
missing = required_columns - set(df.columns)
if missing:
    raise ValueError(f"Missing required columns: {sorted(missing)}")

# Extract features row-by-row.
feature_df = df["text"].apply(extract_features).apply(pd.Series)
featured_df = pd.concat([df, feature_df], axis=1)

# Save for the later model-training stage.
featured_df.to_csv(OUTPUT, index=False)

FEATURES = list(feature_df.columns)

# Distribution plots: histogram + boxplot for every feature.
for feature in FEATURES:
    fig, ax = plt.subplots(figsize=(8, 5))
    for label in ["human", "ai"]:
        values = featured_df.loc[
            featured_df["author"].astype(str).str.lower() == label, feature
        ].dropna()
        if len(values):
            ax.hist(values, bins=30, alpha=0.5, label=label)
    ax.set_title(f"{feature}: Human vs AI")
    ax.set_xlabel(feature)
    ax.set_ylabel("Frequency")
    ax.legend()
    fig.tight_layout()
    fig.savefig(os.path.join(PLOT_DIR, f"{feature}_histogram.png"), dpi=150)
    plt.close(fig)

    fig, ax = plt.subplots(figsize=(7, 5))
    groups = []
    labels = []
    for label in ["human", "ai"]:
        values = featured_df.loc[
            featured_df["author"].astype(str).str.lower() == label, feature
        ].dropna()
        if len(values):
            groups.append(values)
            labels.append(label)
    ax.boxplot(groups, labels=labels)
    ax.set_title(f"{feature}: Human vs AI")
    ax.set_ylabel(feature)
    fig.tight_layout()
    fig.savefig(os.path.join(PLOT_DIR, f"{feature}_boxplot.png"), dpi=150)
    plt.close(fig)

# Numeric summary by class.
summary = featured_df.groupby("author")[FEATURES].agg(["mean", "median", "std"])
summary.to_csv(os.path.join(PLOT_DIR, "feature_summary_by_author.csv"))

# A simple separation score: standardized mean difference.
rows = []
for feature in FEATURES:
    human = featured_df.loc[
        featured_df["author"].astype(str).str.lower() == "human", feature
    ].dropna()
    ai = featured_df.loc[
        featured_df["author"].astype(str).str.lower() == "ai", feature
    ].dropna()

    if len(human) and len(ai):
        pooled_sd = math.sqrt(
            (human.var(ddof=1) + ai.var(ddof=1)) / 2
        )
        effect_size = (
            (ai.mean() - human.mean()) / pooled_sd
            if pooled_sd > 0 else 0.0
        )
        rows.append({
            "feature": feature,
            "human_mean": human.mean(),
            "ai_mean": ai.mean(),
            "difference_ai_minus_human": ai.mean() - human.mean(),
            "standardized_mean_difference": effect_size,
            "absolute_effect_size": abs(effect_size),
        })

separation = pd.DataFrame(rows).sort_values(
    "absolute_effect_size", ascending=False
)
separation.to_csv(
    os.path.join(PLOT_DIR, "feature_separation_scores.csv"), index=False
)

print(f"Loaded rows: {len(df):,}")
print(f"Featured rows: {len(featured_df):,}")
print(f"Saved featured dataset: {OUTPUT}")
print(f"Saved plots and analysis: {PLOT_DIR}/")
print("\nFeature separation ranking:")
print(separation.to_string(index=False))
print("\nReusable example:")
print(extract_features("This is a short example sentence. It has two sentences!"))
